# Running Atlas-CRPS Inference

Deterministic inference using the Atlas-CRPS prognostic model.

This example demonstrates how to run a single-member forecast using the Atlas-CRPS
model, a generative AI weather model that uses a CRPS-trained latent transformer and
an autoencoder to produce 6-hour forecasts on a 0.25 degree global grid. Atlas-CRPS
requires two input lead times (t-6h and t) and maintains an internal latent state for
autoregressive rollouts.

Atlas-CRPS shares its autoencoder with the Atlas (stochastic interpolant) model and
achieves comparable forecast skill, but generates each ensemble member with a single
forward pass instead of an iterative sampling loop, making it substantially faster:
the published Atlas package samples its stochastic interpolant over 100 steps per
6-hour forecast step, so Atlas-CRPS runs with roughly two orders of magnitude fewer
evaluations of the generative core per step (actual end-to-end wall-clock speedup is
somewhat lower once the shared, fixed-cost autoencoder decode is accounted for, but
is still substantial). For most use cases Atlas-CRPS is the recommended entry point;
ensemble members can be generated by calling the model repeatedly from the same
initial condition (see [`torch.manual_seed`][torch.manual_seed] for reproducible members).

[`earth2studio.models.px.Atlas`][earth2studio.models.px.Atlas] and
[`earth2studio.models.px.AtlasCRPS`][earth2studio.models.px.AtlasCRPS] expose the same
user-facing interface, so the two models are interchangeable in code — every usage of
``AtlasCRPS`` below works unchanged if swapped for ``Atlas``, and vice versa.

In this example you will learn:

- How to instantiate the Atlas-CRPS prognostic model
- How to run a single-member forecast with Atlas-CRPS
- How to visualize predicted 10m u-wind and total column water vapour

We need the following:

- Prognostic Model: Use the Atlas-CRPS model [`earth2studio.models.px.AtlasCRPS`][earth2studio.models.px.AtlasCRPS].
- Datasource: Pull data from the ARCO ERA5 data source [`earth2studio.data.ARCO_ERA5`][earth2studio.data.ARCO_ERA5].
- IO Backend: Save the outputs into a Zarr store [`earth2studio.io.ZarrBackend`][earth2studio.io.ZarrBackend].

!!! note
    Atlas-CRPS requires two input lead times (t-6h and t) to produce a forecast.
    The deterministic workflow handles this automatically via the model's
    ``input_coords`` definition.

!!! warning
    Atlas-CRPS was trained on ERA5 data and in-filled NaNs in SST over landmasses with a
    value of 0 K using the ERA5 land-sea mask. If you are using a different SST dataset,
    you will need to in-fill the NaNs using the appropriate land-sea mask.

!!! note
    Atlas-CRPS expects total precipitation as a 6-hour accumulation (``tp06``), not the
    1-hour accumulation returned by a bare ``tp`` request. The model's ``VARIABLES`` list
    already requests ``tp06``, so data sources with a matching lexicon entry (including
    ARCO ERA5) will accumulate the correct window automatically.

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)
# Performance optimization for Atlas-CRPS model
# This is on by default in NGC containers, but other environments may need to set it manually.
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
from dotenv import load_dotenv

load_dotenv()  # TODO: make common example prep function

import numpy as np
import torch

from earth2studio.data import ARCO_ERA5
from earth2studio.data.utils import fetch_data
from earth2studio.io import ZarrBackend
from earth2studio.models.px import AtlasCRPS

# Load the default model package which downloads the checkpoint from HuggingFace
package = AtlasCRPS.load_default_package()
model = AtlasCRPS.load_model(package)

# Create the data source
data = ARCO_ERA5()

# Create the IO handler
io = ZarrBackend()


## Manual Forward Pass
For more control over a *single* forecast step, Atlas-CRPS can be called directly
using [`earth2studio.data.utils.fetch_data`][earth2studio.data.utils.fetch_data] for
initial conditions. This is useful when you need access to intermediate tensors or
want to inspect the raw model output before any further rollout.

!!! warning
    Do not chain repeated `__call__` + `prep_next_input` calls to perform a
    multi-step rollout. Atlas-CRPS keeps an internal low-resolution latent state
    that is only carried forward correctly by `create_iterator`; a direct `__call__`
    always recomputes that latent state from the high-resolution history instead of
    reusing the previous step's prediction, so chaining it across multiple steps is
    scientifically invalid and will silently diverge from the real model rollout.
    Use `create_iterator` (below) for any rollout longer than one step.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Get model input coordinate requirements
input_coords = model.input_coords()

# Fetch initial conditions matching the model's expected variables and lead times
time = np.array([np.datetime64("2024-01-01T00:00")])
x, coords = fetch_data(
    source=data,
    time=time,
    variable=input_coords["variable"],
    lead_time=input_coords["lead_time"],
    device=device,
)

# Add a batch dimension
x = x.unsqueeze(0)
coords["batch"] = np.arange(1)
coords.move_to_end("batch", last=False)

# Single manual forward step (see warning above for why this must not be chained)
y, y_coords = model(x, coords)
lead_hrs = y_coords["lead_time"][0] / np.timedelta64(1, "h")
print(f"Manual step: lead_time = {lead_hrs:.0f}h, shape = {y.shape}")


## Using the model iterator interface
For a less verbose approach, we can use the model iterator interface directly,
which automatically handles the internal latent state during autoregressive rollout.
Simply pass the model and other instantiated components to the workflow, which will
automatically use the iterator interface internally and return outputs to the IO handler.

In [ ]:
import earth2studio.run as run

nsteps = 3
io = run.deterministic(
    ["2024-01-01"],
    nsteps,
    model,
    data,
    io,
    output_coords={"variable": np.array(["u10m", "tcwv"])},
)

print(io.root.tree())


## Post Processing
The last step is to post process our results. We will plot the predicted 10m u-wind
component (u10m) and total column water vapour (tcwv) at 12 hours into the forecast.

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

forecast = "2024-01-01"
step = 2  # lead time = 12 hrs into the forecast

plt.close("all")
projection = ccrs.Robinson()

fig, ax = plt.subplots(1, 2, subplot_kw={"projection": projection}, figsize=(16, 5))

# Plot u10m
im0 = ax[0].pcolormesh(
    io["lon"][:],
    io["lat"][:],
    io["u10m"][0, step],
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    vmin=-20,
    vmax=20,
)
ax[0].set_title(f"{forecast} - u10m - Lead time: {6*step}hrs")
ax[0].coastlines()
ax[0].gridlines()
plt.colorbar(im0, ax=ax[0], shrink=0.6, pad=0.04, label="m/s")

# Plot tcwv
im1 = ax[1].pcolormesh(
    io["lon"][:],
    io["lat"][:],
    io["tcwv"][0, step],
    transform=ccrs.PlateCarree(),
    cmap="Blues",
    vmin=0,
    vmax=70,
)
ax[1].set_title(f"{forecast} - tcwv - Lead time: {6*step}hrs")
ax[1].coastlines()
ax[1].gridlines()
plt.colorbar(im1, ax=ax[1], shrink=0.6, pad=0.04, label="kg/m²")

plt.tight_layout()
plt.savefig("outputs/06_atlas_crps_u10m_tcwv.jpg", dpi=300)
